!pip uninstall torchcrf -y
!pip install pytorch-crf
!pip install -q transformers datasets accelerate seqeval torch
!pip install -q transformers datasets accelerate seqeval torch
!pip install -q git+https://github.com/csebuetnlp/normalizer.git

In [ ]:
%%capture
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
import os
import torch
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'

from unsloth import FastLanguageModel, is_bf16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq, Trainer, EarlyStoppingCallback
from datasets import load_dataset
import glob


# Condifig
PROJECT_DIR      = "Dataset/splits/"
Model = "gemma-3-4b-it" # "llama-3.1-8B-Instruct"
train_file       = os.path.join(PROJECT_DIR, "sft_train_final.jsonl")
eval_file        = os.path.join(PROJECT_DIR, "sft_val_final.jsonl")
output_lora_path = os.path.join("Models/Decoder_Only/" + Model, "lora_" + Model + "_bangla_ee")
checkpoint_dir   = os.path.join("Models/Decoder_Only/" + Model, "training_checkpoints")

max_seq_length = 1024
os.makedirs(checkpoint_dir, exist_ok=True)


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = f"unsloth/{Model}", # gemma-3-12b-it
    max_seq_length = max_seq_length,
    dtype          = None,
    load_in_4bit   = False,
    token          = os.environ.get("HF_TOKEN", None),
)

In [ ]:

# APPLY LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r               = 16, 
    lora_alpha      = 32, # 2r is a common heuristic for lora_alpha
    lora_dropout    = 0,
    bias            = "none",
    target_modules  = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    use_gradient_checkpointing = "unsloth",
)


tokenizer = get_chat_template(tokenizer, chat_template="gemma-3") # gemma-3

pure_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer

def format_and_mask(examples):
    batch = {"input_ids": [], "labels": [], "attention_mask": []}

    for msg in examples["messages"]:
        full_text = pure_tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        prompt_text = pure_tokenizer.apply_chat_template([msg[0]], tokenize=False, add_generation_prompt=True)

        full_tokens = pure_tokenizer(full_text, truncation=True, max_length=max_seq_length, add_special_tokens=False)
        prompt_tokens = pure_tokenizer(prompt_text, truncation=True, max_length=max_seq_length, add_special_tokens=False)

        input_ids = full_tokens["input_ids"]
        prompt_len = len(prompt_tokens["input_ids"])

        labels = [-100] * prompt_len + input_ids[prompt_len:]

        batch["input_ids"].append(input_ids)
        batch["labels"].append(labels)
        batch["attention_mask"].append(full_tokens["attention_mask"])

    return batch


raw_train_dataset = load_dataset("json", data_files=train_file, split="train")
raw_eval_dataset = load_dataset("json", data_files=eval_file, split="train")

train_dataset = raw_train_dataset.map(format_and_mask, batched=True, remove_columns=raw_train_dataset.column_names)
eval_dataset = raw_eval_dataset.map(format_and_mask, batched=True, remove_columns=raw_eval_dataset.column_names)



In [ ]:

training_args = TrainingArguments(
    output_dir = checkpoint_dir,

    per_device_train_batch_size = 16, 
    gradient_accumulation_steps = 4, 

    num_train_epochs  = 5,
    learning_rate     = 1e-4,
    warmup_ratio      = 0.05,

    fp16 = False,
    bf16 = True,

    optim             = "adamw_torch_fused",
    weight_decay      = 0.01,
    lr_scheduler_type = "cosine",
    max_grad_norm     = 1.0,
    logging_steps     = 20,

    eval_strategy          = "steps",
    eval_steps             = 200,
    save_strategy          = "steps",
    save_steps             = 200,
    save_total_limit       = 2,
    load_best_model_at_end = True,
    metric_for_best_model  = "eval_loss",
    greater_is_better      = False,
    report_to              = "none",
)

trainer = Trainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = train_dataset,
    eval_dataset     = eval_dataset,
    args             = training_args,
    data_collator    = DataCollatorForSeq2Seq(
        pure_tokenizer,
        padding=True,
        label_pad_token_id=-100
    ),
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]
)






In [ ]:
# Check if any checkpoint folders exist in the directory
existing_checkpoints = glob.glob(os.path.join(checkpoint_dir, "checkpoint-*"))

if len(existing_checkpoints) > 0:
    print(f"Found existing checkpoints. Resuming training.")
    trainer.train(resume_from_checkpoint=True)
else:
    print("No checkpoints found. Starting training from scratch.")
    trainer.train() # Defaults to resume_from_checkpoint=False


print(f"Saving LoRA adapters to {output_lora_path}")
model.save_pretrained(output_lora_path)
tokenizer.save_pretrained(output_lora_path)